# Games operations sample: load the lakehouse

Loads the synthetic Games operations tables into the default lakehouse as Delta tables and copies the
documents into the lakehouse **Files** area. The tables are the static sources for the Fabric IQ ontology and
the semantic model.

**Before running**
1. Attach a lakehouse (suggested name `GamesLakehouse`) as the default lakehouse of this notebook.
2. Upload the `games-operations` sample folder to **Files/games_ops/** (the folder that contains `generator/`).
3. Choose option A or B in the next cell.

| Option | What happens |
|---|---|
| A. Use pre-generated output | Reads CSV files from `Files/games_ops/output/` produced locally with `generate.py` |
| B. Generate in Fabric | Runs `generate.py` in this notebook and writes the output to `Files/games_ops/output/` |

All data is synthetic. Timestamps are UTC.

In [ ]:
# Parameters
OPTION = "A"                  # "A" = use pre-generated output, "B" = generate here
THEME = "neutral"             # "neutral" or "ioc"
START_DATE = "2026-11-16"     # day 1 of the Games
AS_OF = None                  # e.g. "2026-11-24T11:00"; None = day 9 at 11:00 UTC
SEED = 2026
INCLUDE_FINDINGS_TABLE = False  # True = load ground-truth findings instead of running AI functions extraction

ROOT = "/lakehouse/default/Files/games_ops"
GEN_DIR = f"{ROOT}/generator"
OUT_DIR = f"{ROOT}/output"

## Option B: generate the data in Fabric

In [ ]:
# PDF generation needs reportlab; without it the documents are written as Markdown.
# %pip install reportlab
import sys
if OPTION == "B":
    sys.path.insert(0, GEN_DIR)
    import generate
    args = ["--out", OUT_DIR, "--theme", THEME, "--start-date", START_DATE, "--seed", str(SEED)]
    if AS_OF:
        args += ["--as-of", AS_OF]
    if INCLUDE_FINDINGS_TABLE:
        args += ["--include-findings-table"]
    manifest = generate.main(args)

## Load the lakehouse tables

In [ ]:
import json, os, sys
sys.path.insert(0, GEN_DIR)
from schema import LAKEHOUSE_TABLES
from pyspark.sql import functions as F, types as T

SPARK_TYPES = {"string": T.StringType(), "int": T.IntegerType(), "long": T.LongType(),
               "double": T.DoubleType(), "bool": T.BooleanType(), "date": T.DateType(),
               "timestamp": T.TimestampType()}

spark_path = "Files/games_ops/output/lakehouse"
loaded = {}
for table, cols in LAKEHOUSE_TABLES.items():
    if not os.path.exists(f"{OUT_DIR}/lakehouse/{table}.csv"):
        print(f"skip {table} (no file)")
        continue
    raw = spark.read.option("header", True).csv(f"{spark_path}/{table}.csv")   # all strings, matched by name
    exprs = []
    for c, t, _ in cols:
        src = F.col(c) if c in raw.columns else F.lit(None)
        if t == "timestamp":
            exprs.append(F.to_timestamp(src, "yyyy-MM-dd HH:mm:ss").alias(c))
        elif t == "date":
            exprs.append(F.to_date(src, "yyyy-MM-dd").alias(c))
        elif t == "bool":
            exprs.append((F.lower(src) == "true").alias(c))
        else:
            exprs.append(src.cast(SPARK_TYPES[t]).alias(c))
    df = raw.select(*exprs)
    df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(table)
    loaded[table] = df.count()
    print(f"{table:20s} {loaded[table]:>8,}")


## Check row counts against the manifest

In [ ]:
with open(f"{OUT_DIR}/manifest.json") as f:
    manifest = json.load(f)
problems = [t for t, n in loaded.items() if manifest["row_counts"].get(f"lakehouse.{t}") != n]
print("All row counts match the manifest." if not problems else f"Mismatch: {problems}")
print("Scenario:", json.dumps(manifest["scenario"], indent=2))

## Documents

The PDFs are already in `Files/games_ops/output/documents/` (manuals, inspections, transport, policies).
They feed two paths:
- **Path A**: AI functions extract inspection findings into a `finding` Delta table, bound to the
  ontology as the Finding entity type (next notebook).
- **Path B**: an indexed OneLake knowledge source in Azure AI Search indexes the full text for Foundry IQ.

`document_register` (loaded above) holds the governed metadata, including whether a version is
**Current** or **Superseded**. The PDFs themselves do not say so, as with stale copies in real file shares.

In [ ]:
display(spark.sql("SELECT document_type, document_status, sensitivity, COUNT(*) AS documents "
                  "FROM document_register GROUP BY ALL ORDER BY ALL"))

## Next steps
1. Run `02_eventhouse_setup.kql` in the eventhouse KQL database, then ingest the CSV files from
   `output/eventhouse/` into the four tables.
2. Run `03_build_semantic_model.ipynb` to create the semantic model **Games Operations Model**.
3. Generate the ontology from the semantic model and finish it with the prompts in
   `ontology/ontology-agent-prompts.md`.
